In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../data/players_features.csv")
print(df.shape)

(18404, 25)


In [2]:
y = np.log1p(df["market_value_in_eur"])
X = df.drop(columns=["player_id", "name", "market_value_in_eur"])
X.columns.tolist()

['country_of_citizenship',
 'sub_position',
 'position',
 'foot',
 'height_in_cm',
 'international_caps',
 'international_goals',
 'current_club_domestic_competition_id',
 'caps_missing',
 'age',
 'contract_months_left',
 'no_league',
 'has_contract_info',
 'games',
 'minutes',
 'goals',
 'assists',
 'yellow',
 'red',
 'goals_per90',
 'assists_per90',
 'has_recent_games']

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(X_train.shape, X_test.shape)

(14723, 22) (3681, 22)


In [5]:
num_cols = ["age", "height_in_cm", "international_caps", "international_goals",
            "contract_months_left", "games", "minutes", "goals", "assists",
            "yellow", "goals_per90", "assists_per90"]
cat_cols = ["country_of_citizenship", "position", "sub_position",
            "foot", "current_club_domestic_competition_id"]
bin_cols = ["caps_missing", "no_league", "has_contract_info", "has_recent_games"]

In [6]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=30,
                          sparse_output=False), cat_cols),
    ("bin", "passthrough", bin_cols),
])

In [7]:
results = []

def evaluate(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    mae_eur = mean_absolute_error(np.expm1(y_test), np.expm1(pred))
    results.append({"model": name, "RMSE_log": rmse, "MAE_log": mae,
                    "R2_log": r2, "MAE_EUR": mae_eur})
    return pd.DataFrame(results).round(3)

In [8]:
from sklearn.dummy import DummyRegressor

baseline = Pipeline([("prep", preprocessor),
                     ("model", DummyRegressor(strategy="median"))])
evaluate("Baseline (median)", baseline)

,model,RMSE_log,MAE_log,R2_log,MAE_EUR
0,Baseline (median),1.608,1.258,-0.016,2786903.015


In [9]:
from sklearn.linear_model import LinearRegression

linear = Pipeline([("prep", preprocessor),
                   ("model", LinearRegression())])
evaluate("Linear Regression", linear)

,model,RMSE_log,MAE_log,R2_log,MAE_EUR
0,Baseline (median),1.608,1.258,-0.016,2786903.015
1,Linear Regression,0.940,0.728,0.653,1962992.764


In [10]:
from sklearn.linear_model import Ridge

ridge = Pipeline([("prep", preprocessor),
                  ("model", Ridge(alpha=1.0))])
evaluate("Ridge", ridge)

,model,RMSE_log,MAE_log,R2_log,MAE_EUR
0,Baseline (median),1.608,1.258,-0.016,2786903.015
1,Linear Regression,0.940,0.728,0.653,1962992.764
2,Ridge,0.940,0.728,0.653,1963025.364


In [11]:
from sklearn.linear_model import Lasso

lasso = Pipeline([("prep", preprocessor),
                  ("model", Lasso(alpha=0.001, max_iter=10000))])
evaluate("Lasso", lasso)

,model,RMSE_log,MAE_log,R2_log,MAE_EUR
0,Baseline (median),1.608,1.258,-0.016,2786903.015
1,Linear Regression,0.940,0.728,0.653,1962992.764
2,Ridge,0.940,0.728,0.653,1963025.364
3,Lasso,0.946,0.736,0.648,1970851.983


In [12]:
from sklearn.ensemble import RandomForestRegressor

forest = Pipeline([("prep", preprocessor),
                   ("model", RandomForestRegressor(n_estimators=200,
                                                   random_state=42, n_jobs=-1))])
evaluate("Random Forest", forest)

,model,RMSE_log,MAE_log,R2_log,MAE_EUR
0,Baseline (median),1.608,1.258,-0.016,2786903.015
1,Linear Regression,0.940,0.728,0.653,1962992.764
2,Ridge,0.940,0.728,0.653,1963025.364
3,Lasso,0.946,0.736,0.648,1970851.983
4,Random Forest,0.880,0.674,0.696,1630743.050


In [13]:
from sklearn.ensemble import GradientBoostingRegressor

boost = Pipeline([("prep", preprocessor),
                  ("model", GradientBoostingRegressor(n_estimators=300,
                                                      learning_rate=0.1,
                                                      max_depth=3,
                                                      random_state=42))])
evaluate("Gradient Boosting", boost)

,model,RMSE_log,MAE_log,R2_log,MAE_EUR
0,Baseline (median),1.608,1.258,-0.016,2786903.015
1,Linear Regression,0.940,0.728,0.653,1962992.764
2,Ridge,0.940,0.728,0.653,1963025.364
3,Lasso,0.946,0.736,0.648,1970851.983
4,Random Forest,0.880,0.674,0.696,1630743.050
5,Gradient Boosting,0.846,0.652,0.719,1646288.183


## Results: profile features vs profile + performance features

| Model | R2 (profile only) | R2 (with performance) |
|-------|-------------------|-----------------------|
| Baseline (median) | -0.016 | -0.016 |
| Linear Regression | 0.530 | 0.653 |
| Ridge | 0.530 | 0.653 |
| Lasso | 0.525 | 0.648 |
| Random Forest | 0.588 | 0.696 |
| Gradient Boosting | 0.623 | 0.719 |

Adding last-12-month performance features improved every model. Gradient Boosting
remains the best